In [0]:
file_location = "abfss://dataingestion@projectpractice2.dfs.core.windows.net/userinfo/*.json"

df = (
     spark.read
     .format("json")
     .option("header", "true")
     .option("inferSchema", "true")
     .load(file_location)
)

display(df)
df.createOrReplaceTempView("df")

In [0]:
%sql
drop table projectpractice2.bronze.user_info;

In [0]:
%sql
create table projectpractice2.silver.user_info as
select cast(user_id.user_id as string) as user_id, update_type.update_type,
CAST(from_unixtime(timestamp.timestamp) AS TIMESTAMP) AS converted_timestamp, 
to_date(dob.dob, 'MM/dd/yyyy') as dob, sex.sex, gender.gender, first_name.first_name, last_name.last_name,
parsed.address.street_address, parsed.address.city, parsed.address.state, parsed.address.zip
from (
select 
from_json(value, 'user_id string') as user_id,
from_json(value, 'update_type string') as update_type,
from_json(value, 'timestamp BIGINT') as timestamp,
from_json(value, 'dob string') as dob,
from_json(value, 'sex string') as sex,
from_json(value, 'gender string') as gender,
from_json(value, 'first_name string') as first_name,
from_json(value, 'last_name string') as last_name,
from_json(value, 'address STRUCT<street_address: STRING, city: STRING, state: STRING, zip: STRING>'
  ) AS parsed
FROM df
    )

In [0]:
%sql
select * from projectpractice2.bronze.user_info;

In [0]:
 file_location1 = "abfss://dataingestion@projectpractice2.dfs.core.windows.net/bpm/3-bpm_1.json"

 df1 = (
     spark.read
     .format("json")
     .option("header", "true")
     .option("inferSchema", "true")
     .load(file_location1)
 )

 display(df1)

In [0]:
df1.createOrReplaceTempView("df1")

In [0]:
%sql
select 
value.device_id as device_id,
cast(value.heartrate as decimal(10,2)) as heartrate,
cast(value.time as timestamp) as time
FROM df1

In [0]:
file_location2 = "abfss://dataingestion@projectpractice2.dfs.core.windows.net/workout/4-workout_1.json"

df2 = (
     spark.read
     .format("json")
     .option("header", "true")
     .option("inferSchema", "true")
     .load(file_location2)
)

display(df2)

In [0]:
df2.createOrReplaceTempView("df2")

In [0]:
%sql
create table if not exists projectpractice2.bronze.workout;

In [0]:
(
    df2.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable("projectpractice2.bronze.workout")
)

In [0]:
%sql
create table if not exists projectpractice2.silver.workout;

In [0]:
%sql
create table df22 as
select user_id.user_id, workout_id.workout_id, 
from_unixtime(
    CAST(CAST(timestamp.timestamp AS DOUBLE) AS BIGINT)
) AS timestamp, action.action, session_id.session_id from (select 
from_json(value, 'user_id string') as user_id,
from_json(value, 'workout_id string') as workout_id,
from_json(value, 'timestamp string') as timestamp,
from_json(value, 'action string') as action,
from_json(value, 'session_id string') as session_id
FROM df2)
order by user_id, timestamp

In [0]:
df22 = spark.table("df22")

In [0]:
(
        df22.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable("projectpractice2.silver.workout")
    )

In [0]:
%sql
select * from projectpractice2.silver.workout;

In [0]:
file_location6 = "abfss://dataingestion@projectpractice2.dfs.core.windows.net/gymlogins/5-gym_logins_1.csv"

df6 = (
     spark.read
     .format("csv")
     .option("header", "true")
     .option("inferSchema", "true")
     .load(file_location6)
)

display(df6)

In [0]:
df6.createOrReplaceTempView("df6")

In [0]:
%sql
create table if not exists gym_logins as
select mac_address,count(gym)as ct_gym,
sum(timestampdiff(MINUTE, cast(from_unixtime(login) as timestamp),cast(from_unixtime(logout) as timestamp))) as gym_duration_mins
 from df6 group by mac_address;

In [0]:
%sql
select mac_address,count(gym)as ct_gym,
sum(timestampdiff(MINUTE, cast(from_unixtime(login) as timestamp),cast(from_unixtime(logout) as timestamp))) as gym_duration_mins
 from df6 group by mac_address;

In [0]:
%sql
select * from gym_logins;

In [0]:
%sql
drop table if exists projectpractice2.silver.gym_logins;

In [0]:
%sql
create table if not exists projectpractice2.silver.gym_logins;

In [0]:
# gym_logins is a SQL table (created in Cell 13), so load it as a DataFrame first
gym_logins = spark.table("gym_logins")

(
    gym_logins.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable("projectpractice2.silver.gym_logins")
)

In [0]:
%sql
select * from projectpractice2.silver.gym_logins;

In [0]:
file_location4 = "abfss://dataingestion@projectpractice2.dfs.core.windows.net/datelookup/6-date-lookup.json"

df4 = (
     spark.read
     .format("json")
     .option("header", "true")
     .option("inferSchema", "true")
     .load(file_location4)
)

display(df4)

In [0]:
df4.createOrReplaceTempView("df4")

In [0]:
%sql
select cast(date as date) as date, dayofmonth, dayofweek, dayofyear, month, week, week_part, year from df4;

In [0]:
df3 = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("header", "true")
         #.option("cloudFiles.inferColumnTypes", "true")
         .option("cloudFiles.schemaLocation", "abfss://medallion@projectpractice2.dfs.core.windows.net/checkpoints/bpm/_schemas/")
         .option("checkpointlocation", "abfss://medallion@projectpractice2.dfs.core.windows.net/checkpoints/bpm/_schemas/")
         .load(
             "abfss://dataingestion@projectpractice2.dfs.core.windows.net/bpm/*.json"
         )
         
)

In [0]:
bronze_schema_path = (
    spark.sql("""
        DESCRIBE EXTERNAL LOCATION `medallion2`
    """)
    .selectExpr("concat(url, 'checkpoints/bpm/_schemas/') as url")
    .first()["url"]
)

print(bronze_schema_path)

In [0]:
df3.printSchema()

In [0]:
# Serverless display() forces 'complete' mode on streams, which fails for non-aggregated streams.
# Instead, process all available files into a bronze Delta table, then display it.
query = (
    df3.writeStream
       .format("delta")
       .outputMode("append")
       .option("checkpointLocation", bronze_schema_path.replace("_schemas/", "_checkpoint/"))
       .trigger(availableNow=True)
       .toTable("projectpractice2.bronze.bpm")
)
query.awaitTermination()

display(spark.table("projectpractice2.bronze.bpm"))

In [0]:
display(df3)

In [0]:
file_path = "abfss://medallion@projectpractice2.dfs.core.windows.net/gold/7-gym_summary_1.parquet"

df = (
    spark.read
    .format("parquet")
    .load(file_path)
)

display(df)

In [0]:
file_path1 = "abfss://medallion@projectpractice2.dfs.core.windows.net/gold/8-workout_bpm_summary_2.parquet"

df11 = (
    spark.read
    .format("parquet")
    .load(file_path1)
)

display(df11)